In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
import sys
import numpy as np
sys.path.append('../../../')   # Add parent directory to Python path
import pickle
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

np.random.seed(42)  # For reproducibility


# 1.1 Combine all datasets for training

In [5]:
# Load the curb data (which appears to be stored as a dictionary with scene_0 and scene_1)
with open('../../data/1s_30hz/curb_1s_combined_all.pkl', 'rb') as f:
    curb_data = pickle.load(f)
    data_curb_0 = curb_data['scene_0']
    data_curb_1 = curb_data['scene_1']
    
# Load the other surface types (these appear to be stored as arrays)
with open('../../data/1s_30hz/asphalt_1s_combined_all.pkl', 'rb') as f:
    data_asphalt = pickle.load(f)

with open('../../data/1s_30hz/cobblestone_1s_combined_all.pkl', 'rb') as f:
    data_cobblestone = pickle.load(f)

with open('../../data/1s_30hz/compactgravel_1s_combined_all.pkl', 'rb') as f:
    data_compact_gravel = pickle.load(f)

with open('../../data/1s_30hz/dirt_1s_combined_all.pkl', 'rb') as f:
    data_Dirt = pickle.load(f)

with open('../../data/1s_30hz/pavingstone_1s_combined_all.pkl', 'rb') as f:
    data_PavingStone = pickle.load(f)

with open('../../data/1s_30hz/real_world_1s_combined_3people.pkl', 'rb') as f:
    data_real_world = pickle.load(f)
    data_real_world_0 = data_real_world['scene_0']
    data_real_world_1 = data_real_world['scene_1']

# Print shapes to verify the data was loaded correctly
print("Curb (scene 0):", data_curb_0.shape)
print("Curb (scene 1):", data_curb_1.shape)
print("Asphalt:", data_asphalt.shape)
print("Cobblestone:", data_cobblestone.shape)
print("Compact Gravel:", data_compact_gravel.shape)
print("Dirt:", data_Dirt.shape)
print("Paving Stone:", data_PavingStone.shape)
print("Real World (scene 0):", data_real_world_0.shape)
print("Real World (scene 1):", data_real_world_1.shape)

Curb (scene 0): (617, 30, 3)
Curb (scene 1): (617, 30, 3)
Asphalt: (704, 30, 3)
Cobblestone: (486, 30, 3)
Compact Gravel: (479, 30, 3)
Dirt: (577, 30, 3)
Paving Stone: (645, 30, 3)
Real World (scene 0): (264, 30, 3)
Real World (scene 1): (264, 30, 3)


In [6]:
# Combine curb_1 data from both datasets
combined_curb_1 = np.concatenate([data_curb_1, data_real_world_1])
print(f"Original curb_1 samples: {len(data_curb_1)}")
print(f"Real world curb_1 samples: {len(data_real_world_1)}")
print(f"Combined curb_1 samples: {len(combined_curb_1)}")

Original curb_1 samples: 617
Real world curb_1 samples: 264
Combined curb_1 samples: 881


In [7]:
# Update the curb_1 data creation
curb_1_data = [(segment, "curb_1") for segment in combined_curb_1]
print(f"Total combined curb_1 samples: {len(curb_1_data)}")

# Get count of curb_1 samples to know how many we need from other classes
curb_1_count = len(curb_1_data)

# Create list of other datasets
other_datasets = [
    (data_curb_0, "curb_0"),
    (data_asphalt, "asphalt"),
    (data_cobblestone, "cobblestone"),
    (data_compact_gravel, "compact_gravel"),
    (data_Dirt, "dirt"),
    (data_PavingStone, "paving_stone"),
    (data_real_world_0, "real_world_0"),
]

# Calculate how many samples to take from each other class for even distribution
samples_per_class = curb_1_count // len(other_datasets)
print(f"Taking {samples_per_class} samples from each of the other 6 classes")

# Randomly select samples from other classes
other_class_data = []
for data, label in other_datasets:
    # Randomly select indices
    selected_indices = np.random.choice(len(data), samples_per_class, replace=False)
    # Add selected samples to other_class_data
    for idx in selected_indices:
        other_class_data.append((data[idx], "non_curb"))  # Label all other classes as "non_curb"

print(f"Total non_curb samples: {len(other_class_data)}")

# Combine curb_1 and other class data
combined_dataset = other_class_data + curb_1_data
print(f"Total binary dataset samples: {len(combined_dataset)}")



Total combined curb_1 samples: 881
Taking 125 samples from each of the other 6 classes
Total non_curb samples: 875
Total binary dataset samples: 1756


In [8]:
# Save the binary dataset
with open('../../data/1s_30hz/binary_dataset.pkl', 'wb') as f:
    pickle.dump(combined_dataset, f)

# 1.2 Load data for testing

In [21]:
with open('../../../data/Training/1s_30hz/real_world_1s_combined_3people_test.pkl', 'rb') as f:
    test_data = pickle.load(f)


In [22]:
test_data_0 = test_data['scene_0']
test_data_1 = test_data['scene_1']

# 2. Normalise dataset

## Train

In [ ]:
# Separate data and labels from combined_dataset
X_train = np.array([item[0] for item in combined_dataset])  # Get just the data
y_train = np.array([item[1] for item in combined_dataset])  # Get the labels

In [10]:
X_train_normalized = normalize_3d_data(X_train)

## Test

In [24]:
# Create test dataset with labels (similar to how we created the training dataset)
test_data_curb_1 = [(segment, "curb_1") for segment in test_data_1]
test_data_non_curb = [(segment, "non_curb") for segment in test_data_0]

# Combine curb and non-curb test data
test_dataset = test_data_non_curb + test_data_curb_1

# Separate features and labels
X_test = np.array([item[0] for item in test_dataset])
y_test = np.array([item[1] for item in test_dataset])

# Normalize test data using the same function as training data
X_test_normalized = normalize_3d_data(X_test)

print("Test data shape:", X_test.shape)
print("Number of test samples:", len(y_test))


Test data shape: (494, 30, 3)
Number of test samples: 494


# 4. Labels from string to integer

## Train

In [26]:
# Define custom mapping: curb_1=1, non_curb=0
custom_mapping = {"curb_1": 1, "non_curb": 0}

# # Apply the custom mapping
# y_train_int = np.array([custom_mapping[label] for label in y_train])

# # Create and manually adjust the label_encoder to match your encoding
# label_encoder = LabelEncoder()
# label_encoder.classes_ = np.array(["non_curb", "curb_1"])  # Ensures 0=non_curb, 1=curb_1

# print("Classes:", label_encoder.classes_)
# print("First 10 y_train_int:", y_train_int[:10])
# for idx, label in enumerate(label_encoder.classes_):
#     print(f"{idx}: {label}")

## Test

In [27]:
# Apply the same custom mapping to test labels
y_test_int = np.array([custom_mapping[label] for label in y_test])

# Use the same label_encoder as training (already configured)
print("Test Classes:", label_encoder.classes_)
print("First 10 y_test_int:", y_test_int[:10])

# Print class distribution in test set
unique, counts = np.unique(y_test_int, return_counts=True)
for idx, (label, count) in enumerate(zip(label_encoder.classes_, counts)):
    print(f"{idx}: {label} - {count} samples")

Test Classes: ['non_curb' 'curb_1']
First 10 y_test_int: [0 0 0 0 0 0 0 0 0 0]
0: non_curb - 247 samples
1: curb_1 - 247 samples


## 5: One-hot encode the labels

In [28]:
#y_train_onehot = to_categorical(y_train_int)
y_test_onehot = to_categorical(y_test_int)

#print(y_train_onehot.shape)
print(y_test_onehot.shape)

(494, 2)


In [29]:
# Randomly select an index and check that the one-hot encoding matches the original label
# r = np.random.randint(len(y_train_int))
# assert y_train_onehot[r].argmax() == y_train_int[r]
r = np.random.randint(len(y_test_int))
assert y_test_onehot[r].argmax() == y_test_int[r]

## 6. Save train, test data and labels

In [31]:
# Save test data
with open('../../training_data/X_test_data.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open('../../training_data/y_test_onehot.pkl', 'wb') as f:
    pickle.dump(y_test_onehot, f)

## 6. Train, validation Spilt

In [16]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized,
    y_train_onehot,
    test_size=0.2,           # 20% for validation
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (1404, 30, 3) (1404, 2)
Validation shape: (352, 30, 3) (352, 2)


In [17]:
# Save training and validation data
with open('../../data/1s_30hz/TrainTest_2class/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('../../data/1s_30hz/TrainTest_2class/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val, f)

with open('../../data/1s_30hz/TrainTest_2class/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('../../data/1s_30hz/TrainTest_2class/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val, f)